In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="../data/sample_texts.txt",
    column_names=["text", "label"]
)

print(dataset["train"][:5])

{'text': ['I love deep learning', 'Transformers are powerful models', 'NLP is fascinating', 'I hate bugs in software', 'This code is terrible'], 'label': [1, 1, 1, 0, 0]}


In [4]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

In [6]:
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print(dataset["train"][0])

{'labels': tensor(1), 'input_ids': tensor([ 101, 1045, 2293, 2784, 4083,  102,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])}


In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_dir="./logs",
    logging_steps=1,
    report_to="none"
)

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

In [10]:
trainer.train()

  0%|          | 0/12 [00:00<?, ?it/s]

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 0.7008, 'grad_norm': 2.556992769241333, 'learning_rate': 4.5833333333333334e-05, 'epoch': 0.25}
{'loss': 0.665, 'grad_norm': 2.4463531970977783, 'learning_rate': 4.166666666666667e-05, 'epoch': 0.5}
{'loss': 0.736, 'grad_norm': 2.9063808917999268, 'learning_rate': 3.7500000000000003e-05, 'epoch': 0.75}
{'loss': 0.6345, 'grad_norm': 5.6119465827941895, 'learning_rate': 3.3333333333333335e-05, 'epoch': 1.0}
{'loss': 0.5691, 'grad_norm': 2.9550747871398926, 'learning_rate': 2.916666666666667e-05, 'epoch': 1.25}
{'loss': 0.6267, 'grad_norm': 3.2819032669067383, 'learning_rate': 2.5e-05, 'epoch': 1.5}
{'loss': 0.5574, 'grad_norm': 5.031549453735352, 'learning_rate': 2.0833333333333336e-05, 'epoch': 1.75}
{'loss': 0.4531, 'grad_norm': 5.3559250831604, 'learning_rate': 1.6666666666666667e-05, 'epoch': 2.0}
{'loss': 0.5644, 'grad_norm': 3.4039525985717773, 'learning_rate': 1.25e-05, 'epoch': 2.25}
{'loss': 0.5671, 'grad_norm': 3.4608094692230225, 'learning_rate': 8.333333333333334e-06

TrainOutput(global_step=12, training_loss=0.5892935742934545, metrics={'train_runtime': 8.0142, 'train_samples_per_second': 2.62, 'train_steps_per_second': 1.497, 'total_flos': 173863460736.0, 'train_loss': 0.5892935742934545, 'epoch': 3.0})